# Clasificación de imágenes satelitales con Deep Learning

**Proyecto individual | TensorFlow + Keras | Dataset EuroSAT RGB**

**Objetivo general:** entrenar, evaluar y mejorar una red neuronal convolucional (CNN) para clasificar imágenes de cobertura terrestre.

### ¿Qué contiene este notebook?
1. Introducción: problema y relevancia
2. Datos: fuente, exploración, separación y preprocesamiento
3. Desarrollo del modelo: primera CNN y evaluación con validación
4. Ajuste del modelo: segunda CNN, técnicas de regularización y comparación
5. Resultados: evaluación final única sobre prueba
6. Conclusiones: interpretación de métricas reales y limitaciones
7. Aplicación: predicción sobre imágenes nuevas, exportación `.keras`

> **Instrucciones:** ejecuta las celdas **en orden** en Google Colab. Se recomienda `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`, si está disponible. No se incluyen métricas inventadas: se producen durante el entrenamiento. Al finalizar, guarda el notebook **con sus salidas** en GitHub y sube `modelo_final.keras`.

> **Condición de la consigna:** confirma antes de entregar que EuroSAT no se utilizó en tu clase. Este dataset no es un *toy dataset* como MNIST.

## 0. Preparación del entorno

Usaremos bibliotecas estándar de Colab. Si falta `tensorflow_datasets`, la primera celda lo instalará; el resto del proyecto se ejecuta en la misma sesión.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("tensorflow_datasets") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-datasets"])

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
BATCH_SIZE = 64
EPOCAS_BASE = 10
EPOCAS_MEJORADO = 15

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("TensorFlow Datasets:", tfds.__version__)
print("GPUs disponibles:", tf.config.list_physical_devices("GPU"))
print("Semilla:", SEED)

## 1. Introducción

### 1.1 Planteamiento del problema

Distinguir tipos de cobertura terrestre a partir de fotografías satelitales es relevante para el monitoreo ambiental, el análisis del territorio y la planeación urbana. La clasificación manual de grandes colecciones de imágenes exige mucho tiempo; el aprendizaje profundo permite automatizar parte de este proceso.

Este proyecto aborda un **problema de clasificación multiclase**: dada una imagen satelital RGB de 64 × 64 píxeles, predecir una de diez categorías de uso/cobertura de suelo.

### 1.2 Objetivos específicos

- Cargar y explorar las clases e imágenes del dataset EuroSAT RGB.
- Crear un conjunto de **entrenamiento**, uno de **validación** y otro de **prueba** sin utilizar prueba para ajustar el modelo.
- Entrenar una CNN inicial y evaluar su error de entrenamiento/validación.
- Entrenar una segunda CNN con cambios explícitos de arquitectura y regularización.
- Seleccionar el mejor modelo **solo con validación** y medir una sola vez su desempeño en prueba.
- Guardar arquitectura y pesos en `modelo_final.keras` y usarlo para realizar predicciones.

### 1.3 Fuente y descripción de los datos

**Dataset:** EuroSAT RGB, basado en imágenes de Sentinel-2. Contiene **27,000 muestras etiquetadas** de **64 × 64 píxeles** con **3 canales de color (RGB)** y **10 categorías**. Es diferente de MNIST y de los conjuntos de juguetes habituales.

**Fuente oficial:** https://www.tensorflow.org/datasets/catalog/eurosat

**Repositorio original:** https://github.com/phelber/eurosat

**Referencia:** Helber, P., Bischke, B., Dengel, A., & Borth, D. (2017). *EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification*. arXiv:1709.00029.

**Limitación metodológica anticipada:** las imágenes satelitales pueden contener patrones geográficos similares. Una separación aleatoria estratificada permite una evaluación razonable del ejercicio, pero no sustituye una validación geográficamente independiente.

## 2. Datos: adquisición, exploración y preprocesamiento

### 2.1 Descarga del dataset

TensorFlow Datasets publica EuroSAT RGB bajo el identificador `eurosat/rgb`. No es necesario subir los datos al repositorio: este notebook los descarga de su fuente pública cuando se ejecuta.

In [ ]:
datos_tf, info = tfds.load(
    "eurosat/rgb",
    split="train",
    as_supervised=True,
    with_info=True,
    shuffle_files=False
)

nombres_clases = info.features["label"].names
num_clases = len(nombres_clases)

print("Total de observaciones:", info.splits["train"].num_examples)
print("Forma de imagen:", info.features["image"].shape)
print("Número de clases:", num_clases)
print("Clases:")
for i, clase in enumerate(nombres_clases):
    print(f"  {i}: {clase}")

### 2.2 Convertir a arreglos y comprobar calidad básica

El conjunto completo cabe razonablemente en la RAM de Colab en formato `uint8` (aproximadamente 332 MB de píxeles, sin contar copias). Esto simplifica una separación **estratificada**, para mantener las proporciones de clases en cada subconjunto. Para conjuntos realmente grandes convendría trabajar por archivos/streams.

In [ ]:
imagenes_lista = []
etiquetas_lista = []

for imagen, etiqueta in tfds.as_numpy(datos_tf):
    imagenes_lista.append(imagen)
    etiquetas_lista.append(int(etiqueta))

X = np.stack(imagenes_lista).astype(np.uint8)
y = np.asarray(etiquetas_lista, dtype=np.int64)
del imagenes_lista, etiquetas_lista

print("Imágenes:", X.shape, "Tipo:", X.dtype)
print("Etiquetas:", y.shape, "Tipo:", y.dtype)
print("Valores de píxeles (mínimo, máximo):", X.min(), X.max())
print("Etiquetas únicas:", np.unique(y))
assert len(X) == len(y) == 27000
assert X.shape[1:] == (64, 64, 3)
assert len(np.unique(y)) == num_clases == 10

### 2.3 Distribución por categoría

Antes de entrenar, revisamos si alguna clase tiene muchas menos muestras. Si las clases están desbalanceadas, la exactitud total por sí sola podría ocultar errores en categorías minoritarias.

In [ ]:
conteos = np.bincount(y, minlength=num_clases)
distribucion = pd.DataFrame({
    "Categoría": nombres_clases,
    "Cantidad": conteos,
    "Porcentaje": np.round(100 * conteos / len(y), 2)
}).sort_values("Cantidad", ascending=False)
print(distribucion.to_string(index=False))

plt.figure(figsize=(11, 4))
plt.bar(distribucion["Categoría"], distribucion["Cantidad"])
plt.title("Cantidad de imágenes por categoría")
plt.ylabel("Número de imágenes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 2.4 Visualización de ejemplos

Las imágenes son RGB. Su resolución es suficiente para aprender patrones de textura, color y organización espacial, aunque puede limitar el reconocimiento de estructuras pequeñas.

In [ ]:
fig, ejes = plt.subplots(2, 5, figsize=(13, 6))
for clase_id, ax in enumerate(ejes.ravel()):
    idx = int(np.flatnonzero(y == clase_id)[0])
    ax.imshow(X[idx])
    ax.set_title(nombres_clases[clase_id])
    ax.axis("off")
plt.suptitle("Una imagen por categoría", y=1.02)
plt.tight_layout()
plt.show()

### 2.5 División estratificada: 70% / 15% / 15%

- **Entrenamiento (70%):** utilizado para actualizar los pesos de las redes.
- **Validación (15%):** utilizado para comparar versiones y decidir qué arquitectura conservar.
- **Prueba (15%):** se mantiene **sin consultar durante el ajuste** y se usa al final, una sola vez, para evaluar el modelo seleccionado.

`stratify` preserva aproximadamente la composición de categorías en los tres grupos. La semilla fija hace que las divisiones sean reproducibles.

In [ ]:
X_train, X_temporal, y_train, y_temporal = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temporal, y_temporal, test_size=0.50, random_state=SEED, stratify=y_temporal
)
del X, y, X_temporal, y_temporal

for nombre, imagenes, etiquetas in [
    ("Entrenamiento", X_train, y_train),
    ("Validación", X_val, y_val),
    ("Prueba", X_test, y_test)
]:
    print(f"{nombre}: {len(etiquetas):,} imágenes ({len(etiquetas)/27000:.1%})")
    assert len(np.unique(etiquetas)) == num_clases

assert len(X_train) == 18900
assert len(X_val) == 4050
assert len(X_test) == 4050

In [ ]:
filas = []
for i, nombre in enumerate(nombres_clases):
    filas.append({
        "Categoría": nombre,
        "Entrenamiento": int(np.sum(y_train == i)),
        "Validación": int(np.sum(y_val == i)),
        "Prueba": int(np.sum(y_test == i))
    })
print(pd.DataFrame(filas).to_string(index=False))

### 2.6 Pipeline de datos y normalización

Las imágenes se mantienen originalmente como `uint8` (valores entre 0 y 255). **Dentro de cada modelo** utilizamos una capa `Rescaling(1./255)` que normaliza los píxeles a aproximadamente [0, 1]. Esto garantiza que la misma transformación se aplique durante entrenamiento, evaluación e inferencia.

El *shuffling* se aplica **únicamente** a entrenamiento; validación y prueba permanecen en orden estable para asociar etiquetas y predicciones correctamente.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=5000, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

lote_imagenes, lote_etiquetas = next(iter(train_ds))
print("Forma de un lote de imágenes:", lote_imagenes.shape)
print("Forma de un lote de etiquetas:", lote_etiquetas.shape)
print("Tipo de imágenes:", lote_imagenes.dtype)

## 3. Desarrollo del modelo: CNN inicial

### 3.1 Arquitectura

La primera red utiliza tres bloques de **convolución + max pooling**, seguidos de una capa `Flatten` y capas densas para clasificar. Por lo tanto, **no está formada únicamente por capas densas** y satisface la restricción de usar capas especializadas.

- `Conv2D` aprende patrones visuales locales (bordes, colores, texturas).
- `MaxPooling2D` reduce la resolución espacial conservando activaciones relevantes.
- `Flatten` transforma los mapas de características en un vector.
- `Dense(10, softmax)` devuelve probabilidades para las diez categorías.

Se emplea **sparse categorical crossentropy** porque las etiquetas son números enteros de 0 a 9.

In [ ]:
def crear_cnn_base():
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(64, 64, 3)),
        tf.keras.layers.Rescaling(1.0 / 255),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(num_clases, activation="softmax")
    ], name="cnn_base")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

modelo_base = crear_cnn_base()
modelo_base.summary()

### 3.2 Entrenamiento inicial

Usamos `EarlyStopping` para detener el proceso si la pérdida de validación deja de mejorar, y `ModelCheckpoint` para conservar el modelo correspondiente a la **menor pérdida de validación**. Este checkpoint se recarga antes de comparar modelos.

In [ ]:
callbacks_base = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "modelo_base.keras", monitor="val_loss", save_best_only=True
    )
]

historia_base = modelo_base.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCAS_BASE,
    callbacks=callbacks_base,
    verbose=1
)
print("Épocas realizadas (base):", len(historia_base.history["loss"]))

### 3.3 Gráficas de aprendizaje del modelo inicial

La diferencia entre entrenamiento y validación ayuda a detectar problemas: por ejemplo, si el modelo mejora mucho en entrenamiento pero empeora en validación, podría estar sobreajustándose.

In [ ]:
def graficar_historial(historia, titulo):
    h = historia.history
    epocas = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(epocas, h["accuracy"], label="Entrenamiento")
    ax[0].plot(epocas, h["val_accuracy"], label="Validación")
    ax[0].set(title=f"{titulo}: accuracy", xlabel="Época", ylabel="Accuracy")
    ax[0].legend()
    ax[1].plot(epocas, h["loss"], label="Entrenamiento")
    ax[1].plot(epocas, h["val_loss"], label="Validación")
    ax[1].set(title=f"{titulo}: loss", xlabel="Época", ylabel="Loss")
    ax[1].legend()
    plt.tight_layout()
    plt.show()

graficar_historial(historia_base, "Modelo inicial")

In [ ]:
modelo_base = tf.keras.models.load_model("modelo_base.keras")
metricas_base_val = modelo_base.evaluate(val_ds, return_dict=True, verbose=0)
print("VALIDACIÓN — MODELO INICIAL")
for metrica, valor in metricas_base_val.items():
    print(f"{metrica}: {valor:.4f}")
print("Nota: el conjunto de prueba aún no se ha utilizado.")

## 4. Ajuste del modelo: CNN mejorada

### 4.1 Cambios implementados

Entrenaremos una **segunda versión** (mínimo una iteración de mejora, como exige la consigna):

1. **Aumento de datos (*data augmentation*):** volteos, rotaciones moderadas y zoom durante entrenamiento, para exponer a la red a variantes visuales plausibles. Las capas aleatorias están dentro del modelo y permanecen inactivas en inferencia/evaluación.
2. **Batch Normalization:** puede estabilizar las distribuciones de activaciones durante el aprendizaje.
3. **Dropout:** desactiva temporalmente una fracción de activaciones durante entrenamiento y puede disminuir el sobreajuste.
4. **Global Average Pooling:** sustituye `Flatten`, lo que reduce mucho la cantidad de parámetros de la cabeza de clasificación.
5. **ReduceLROnPlateau:** reduce el *learning rate* si se estanca la pérdida de validación.

**Importante:** cambiar varias cosas simultáneamente permite comparar dos versiones completas, **pero no demostrar cuál componente individual produjo una mejora**. Para atribuciones causales habría que hacer experimentos de ablación, añadiendo un cambio por vez.

In [ ]:
def crear_cnn_mejorada():
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(64, 64, 3)),
        tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        tf.keras.layers.RandomRotation(0.10, seed=SEED),
        tf.keras.layers.RandomZoom(0.10, seed=SEED),
        tf.keras.layers.Rescaling(1.0 / 255),

        tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.15),

        tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.20),

        tf.keras.layers.Conv2D(128, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.25),

        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.40),
        tf.keras.layers.Dense(num_clases, activation="softmax")
    ], name="cnn_mejorada")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

modelo_mejorado = crear_cnn_mejorada()
modelo_mejorado.summary()
print("Parámetros del modelo base:", modelo_base.count_params())
print("Parámetros del modelo mejorado:", modelo_mejorado.count_params())

### 4.2 Entrenamiento de la segunda versión

Se conservan el mismo dataset, la misma división, la misma métrica y la misma función de pérdida; varían la arquitectura, regularización y programación del *learning rate*. Esto permite una comparación práctica aunque no una atribución aislada de cada cambio.

In [ ]:
callbacks_mejorado = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2,
        min_lr=1e-5, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "modelo_mejorado.keras", monitor="val_loss", save_best_only=True
    )
]

historia_mejorado = modelo_mejorado.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCAS_MEJORADO,
    callbacks=callbacks_mejorado,
    verbose=1
)
print("Épocas realizadas (ajustado):", len(historia_mejorado.history["loss"]))

In [ ]:
graficar_historial(historia_mejorado, "Modelo mejorado")

### 4.3 Comparación objetiva y selección del modelo

Se selecciona el mejor candidato con la **accuracy de validación** de los checkpoints recargados; en caso de empate, gana el de menor `val_loss`. No se consulta el conjunto de prueba para decidir.

In [ ]:
modelo_mejorado = tf.keras.models.load_model("modelo_mejorado.keras")
metricas_mejorado_val = modelo_mejorado.evaluate(val_ds, return_dict=True, verbose=0)

comparacion = pd.DataFrame([
    {
        "Modelo": "CNN inicial",
        "Accuracy validación": metricas_base_val["accuracy"],
        "Loss validación": metricas_base_val["loss"],
        "Parámetros": modelo_base.count_params(),
        "Épocas": len(historia_base.history["loss"])
    },
    {
        "Modelo": "CNN ajustada",
        "Accuracy validación": metricas_mejorado_val["accuracy"],
        "Loss validación": metricas_mejorado_val["loss"],
        "Parámetros": modelo_mejorado.count_params(),
        "Épocas": len(historia_mejorado.history["loss"])
    }
])
print(comparacion.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

ranking = comparacion.sort_values(
    ["Accuracy validación", "Loss validación"],
    ascending=[False, True]
)
nombre_ganador = ranking.iloc[0]["Modelo"]
modelo_final = modelo_mejorado if nombre_ganador == "CNN ajustada" else modelo_base

diferencia = metricas_mejorado_val["accuracy"] - metricas_base_val["accuracy"]
print(f"\nModelo seleccionado con validación: {nombre_ganador}")
print(f"Cambio en accuracy de validación (ajustado - inicial): {diferencia:+.4%} (puntos porcentuales)")
if diferencia > 0.001:
    print("La versión ajustada mejora la accuracy de validación en esta ejecución.")
elif diferencia < -0.001:
    print("La versión ajustada NO mejora la accuracy de validación en esta ejecución.")
else:
    print("El cambio en accuracy de validación es muy pequeño.")
print("No es posible identificar la contribución individual de cada cambio con solo estos dos experimentos.")

## 5. Resultados: evaluación final sobre el conjunto de prueba

**A partir de este punto** consultamos el conjunto de prueba. Se utiliza solo el modelo ganador seleccionado con validación, sin modificarlo ni volver a optimizarlo con estas observaciones.

Reportamos *accuracy*, *loss*, *precision*, *recall* y *F1-score* por categoría, así como una matriz de confusión. La matriz permite identificar qué clases son más difíciles de distinguir.

In [ ]:
metricas_test = modelo_final.evaluate(test_ds, return_dict=True, verbose=0)
probabilidades = modelo_final.predict(test_ds, verbose=0)
predicciones = np.argmax(probabilidades, axis=1)
verdaderas = y_test.copy()

f1_macro = f1_score(verdaderas, predicciones, average="macro", zero_division=0)
print("MODELO FINAL:", nombre_ganador)
print(f"Accuracy en prueba: {metricas_test['accuracy']:.4f}")
print(f"Loss en prueba: {metricas_test['loss']:.4f}")
print(f"F1 macro en prueba: {f1_macro:.4f}")
print("\nREPORTE POR CATEGORÍA")
print(classification_report(
    verdaderas, predicciones,
    labels=np.arange(num_clases),
    target_names=nombres_clases,
    digits=3, zero_division=0
))

In [ ]:
matriz = confusion_matrix(
    verdaderas, predicciones, labels=np.arange(num_clases)
)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(matriz, cmap="Blues")
ax.set_xticks(range(num_clases), nombres_clases, rotation=55, ha="right")
ax.set_yticks(range(num_clases), nombres_clases)
ax.set_xlabel("Predicción")
ax.set_ylabel("Etiqueta real")
ax.set_title(f"Matriz de confusión — {nombre_ganador}")
for i in range(num_clases):
    for j in range(num_clases):
        valor = matriz[i, j]
        ax.text(j, i, str(valor), ha="center", va="center",
                color="white" if valor > matriz.max() / 2 else "black", fontsize=8)
plt.colorbar(im, ax=ax, label="Número de imágenes")
plt.tight_layout()
plt.show()

In [ ]:
reporte = classification_report(
    verdaderas, predicciones, labels=np.arange(num_clases),
    target_names=nombres_clases, output_dict=True, zero_division=0
)
f1_clases = pd.DataFrame([
    {"Categoría": clase, "F1": reporte[clase]["f1-score"],
     "Recall": reporte[clase]["recall"], "Precision": reporte[clase]["precision"]}
    for clase in nombres_clases
]).sort_values("F1")
print("Categorías ordenadas por F1 (más difíciles primero):")
print(f1_clases.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

plt.figure(figsize=(10, 4))
plt.bar(f1_clases["Categoría"], f1_clases["F1"])
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.title("F1-score por categoría (prueba)")
plt.ylabel("F1-score")
plt.tight_layout()
plt.show()

### 5.1 Ejemplos de aciertos y errores

Esta visualización complementa las métricas: revisar imágenes equivocadas permite formular hipótesis sobre similitud visual, límites de resolución o confusión entre categorías, pero no basta por sí sola para identificar la causa de un error.

In [ ]:
indices_fallos = np.flatnonzero(predicciones != verdaderas)
indices_aciertos = np.flatnonzero(predicciones == verdaderas)
seleccion = np.concatenate([indices_fallos[:6], indices_aciertos[:6]])

fig, ejes = plt.subplots(2, 6, figsize=(16, 6))
for ax, idx in zip(ejes.ravel(), seleccion):
    ax.imshow(X_test[idx])
    bien = predicciones[idx] == verdaderas[idx]
    ax.set_title(
        f"Real: {nombres_clases[verdaderas[idx]]}\n"
        f"Pred: {nombres_clases[predicciones[idx]]}\n"
        f"Conf: {probabilidades[idx].max():.1%} {'✓' if bien else '✗'}",
        fontsize=8
    )
    ax.axis("off")
for ax in ejes.ravel()[len(seleccion):]:
    ax.axis("off")
plt.suptitle("Arriba: errores disponibles | Abajo: aciertos disponibles", y=1.02)
plt.tight_layout()
plt.show()
print("Errores en test:", len(indices_fallos), "de", len(verdaderas))

## 6. Conclusiones

La celda siguiente construye un **resumen cuantitativo con métricas de esta ejecución** para evitar inventar resultados. Después de ejecutarla, complementa las conclusiones en esta sección con **2–4 frases propias** que interpreten qué confusiones observaste en la matriz y qué limitaciones consideras más importantes.

### Aspectos que deben discutirse

- ¿Cuál versión ganó en validación? ¿La regularización/arquitectura más compleja realmente mejoró las métricas?
- ¿Qué clases tienen menor F1? ¿Qué pares se confunden más en la matriz?
- ¿Qué muestra la diferencia entre entrenamiento y validación?
- ¿Qué harías a continuación? Por ejemplo, probar *transfer learning*, más épocas, optimizar una técnica a la vez, calibrar probabilidades o construir un split geográfico.

> Una ganancia observada entre dos ejecuciones **no demuestra** por sí sola que cada una de las técnicas añadidas sea responsable. Para eso se requieren comparaciones controladas.

In [ ]:
peor_clase = f1_clases.iloc[0]
mejor_clase = f1_clases.iloc[-1]
mejor_val = max(
    metricas_base_val["accuracy"],
    metricas_mejorado_val["accuracy"]
)
print("CONCLUSIONES GENERADAS A PARTIR DE ESTA EJECUCIÓN")
print(f"1. El modelo seleccionado fue {nombre_ganador}, con accuracy de validación {mejor_val:.2%}.")
print(f"2. En el conjunto de prueba logró accuracy {metricas_test['accuracy']:.2%} y F1 macro {f1_macro:.3f}.")
print(f"3. La categoría con F1 más bajo fue '{peor_clase['Categoría']}' ({peor_clase['F1']:.3f}), mientras que la mejor fue '{mejor_clase['Categoría']}' ({mejor_clase['F1']:.3f}).")
if diferencia > 0:
    print(f"4. La CNN ajustada superó a la inicial en validación por {100*diferencia:.2f} puntos porcentuales.")
elif diferencia < 0:
    print(f"4. La CNN ajustada obtuvo {100*abs(diferencia):.2f} puntos porcentuales MENOS que la inicial en validación; más regularización no aseguró una mejora.")
else:
    print("4. Ambas CNN obtuvieron la misma accuracy de validación.")
print("5. Limitación: al cambiar varios componentes simultáneamente, no se conoce su efecto individual; además, no se midió generalización a otra región geográfica.")
print("6. Próxima mejora sugerida: realizar una ablación de técnicas o validar en escenas de otra región.")

## 7. Aplicación: predicciones y entrega del modelo

### 7.1 Guardar arquitectura y pesos en `.keras`

El formato `.keras` conserva la arquitectura, los pesos y la configuración del modelo. También guardamos métricas de validación/prueba en un JSON para documentar el resultado. El archivo `.keras` es un **entregable obligatorio** junto con el notebook.

In [ ]:
ruta_modelo = "modelo_final.keras"
modelo_final.save(ruta_modelo)

reporte_final = {
    "dataset": "eurosat/rgb",
    "seed": SEED,
    "train_samples": len(y_train),
    "val_samples": len(y_val),
    "test_samples": len(y_test),
    "selected_model": nombre_ganador,
    "validation_baseline": {k: float(v) for k, v in metricas_base_val.items()},
    "validation_improved": {k: float(v) for k, v in metricas_mejorado_val.items()},
    "test": {**{k: float(v) for k, v in metricas_test.items()}, "f1_macro": float(f1_macro)},
    "classes": nombres_clases
}
with open("metricas_finales.json", "w", encoding="utf-8") as archivo:
    json.dump(reporte_final, archivo, indent=2, ensure_ascii=False)

print("Modelo guardado:", os.path.abspath(ruta_modelo))
print("Tamaño (MB):", round(os.path.getsize(ruta_modelo) / (1024**2), 2))
print("Métricas guardadas: metricas_finales.json")

### 7.2 Comprobar que el modelo guardado puede volver a cargarse

Esto demuestra que la solución no depende únicamente de los pesos que quedaron en memoria durante el entrenamiento.

In [ ]:
modelo_recargado = tf.keras.models.load_model("modelo_final.keras")
original = modelo_final.predict(X_test[:5], verbose=0)
recargado = modelo_recargado.predict(X_test[:5], verbose=0)
np.testing.assert_allclose(original, recargado, rtol=1e-4, atol=1e-5)
print("OK: el modelo .keras se recargó y reproduce las predicciones.")
print("Categorías predichas:", [nombres_clases[i] for i in np.argmax(recargado, axis=1)])

### 7.3 Función para clasificar una imagen nueva

La función recibe la **ruta de un archivo** de imagen RGB, ajusta su tamaño a 64 × 64 píxeles, calcula probabilidades y muestra las tres categorías principales. El modelo ya contiene la normalización; **no se divide entre 255 otra vez**.

**Advertencia:** la confianza reportada es una salida `softmax`, no una garantía de que la predicción sea correcta; fotografías no satelitales, con distinta resolución o fuera del dominio EuroSAT pueden producir predicciones poco fiables.

In [ ]:
def clasificar_imagen(ruta, modelo=modelo_recargado):
    imagen = tf.keras.utils.load_img(
        ruta, target_size=(64, 64), color_mode="rgb"
    )
    arr = tf.keras.utils.img_to_array(imagen)
    probs = modelo.predict(np.expand_dims(arr, axis=0), verbose=0)[0]
    indices = np.argsort(probs)[::-1][:3]
    resultado = [(nombres_clases[i], float(probs[i])) for i in indices]
    print("Predicción más probable:", resultado[0][0])
    for clase, prob in resultado:
        print(f"  {clase}: {prob:.2%}")
    plt.figure(figsize=(4, 4))
    plt.imshow(imagen)
    plt.title(f"Predicción: {resultado[0][0]}")
    plt.axis("off")
    plt.show()
    return resultado

# Demostración verificable sin necesitar un archivo externo:
ruta_ejemplo = "ejemplo_eurosat.png"
tf.keras.utils.save_img(ruta_ejemplo, X_test[0])
print("Imagen de ejemplo guardada:", ruta_ejemplo)
resultado_ejemplo = clasificar_imagen(ruta_ejemplo)
print("Etiqueta verdadera:", nombres_clases[y_test[0]])

### 7.4 Probar una imagen externa (opcional)

En Colab, ejecuta la siguiente celda, selecciona una imagen desde tu equipo y la función realizará su predicción. Para una prueba significativa, procura que sea una imagen satelital RGB similar a EuroSAT.

In [ ]:
# Cambia a True cuando quieras subir y probar una imagen externa.
PROBAR_IMAGEN_EXTERNA = False

if PROBAR_IMAGEN_EXTERNA:
    from google.colab import files
    archivos = files.upload()
    if archivos:
        ruta_subida = next(iter(archivos))
        clasificar_imagen(ruta_subida)
else:
    print("Prueba externa desactivada. Cambia PROBAR_IMAGEN_EXTERNA = True para subir una imagen.")

### 7.5 Descargar los artefactos finales desde Colab

Activa la siguiente celda cuando termines. Descarga el `.keras` y el JSON para subirlos manualmente a GitHub. El notebook se guarda con **Archivo → Guardar una copia en GitHub**, procurando conservar todas las salidas ejecutadas.

In [ ]:
# Cambia a True para descargar archivos desde Colab.
DESCARGAR_ENTREGABLES = False

if DESCARGAR_ENTREGABLES:
    from google.colab import files
    files.download("modelo_final.keras")
    files.download("metricas_finales.json")
else:
    print("Archivos listos en la sesión:",
          [f for f in ["modelo_final.keras", "metricas_finales.json"] if os.path.exists(f)])
    print("Cambia DESCARGAR_ENTREGABLES = True para descargarlos.")

---
## Checklist de entrega

- [ ] Repositorio de GitHub **público** con enlace en Canvas.
- [ ] Notebook `.ipynb` **con todas las celdas ejecutadas y sus salidas**.
- [ ] Secciones completas: Introducción, Datos, Desarrollo, Ajuste, Resultados, Conclusiones y Aplicación.
- [ ] Dataset distinto a los vistos en clase y no considerado *toy dataset*.
- [ ] Comparación de al menos dos versiones del modelo.
- [ ] Evaluación final con conjunto de prueba separado.
- [ ] Archivo `modelo_final.keras` subido y comprobado.
- [ ] Función de predicción demostrada.
- [ ] Revisar y completar la interpretación personal de la sección de Conclusiones.

**Consejo de reproducibilidad:** si una celda marca error, evita ejecutar bloques desordenados: corrige el error y usa `Entorno de ejecución → Ejecutar todo` para reconstruir los resultados de forma consistente.